# 0. Library

In [36]:
import pandas as pd
from pathlib import Path
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp
from features import general_func as gf
import constants_data as cd

# Reload
importlib.reload(du)
importlib.reload(dp)
importlib.reload(gf)
importlib.reload(cd)

<module 'constants_data' from '/home/smira/myproject/detection_AD_with_VR_data/src/notebooks/../constants_data.py'>

# 1. Paths and Constants

In [37]:
# Constants and paths
parent_folder = Path("../..") # go 2 folder up= "../.."
df_path = parent_folder / "data" / "produced_csv" / "1.extracted_features.csv"
 
df = pd.read_csv(df_path)

df.head()

,Paitent_id,Age,Gender,dominant_hand,Sessions_Completed_out_of_4,Help_Rating_out_of_5,MoCA_Score,Tutorial_reading_time_happened,Tutorial_total_reading_time,Tutorial_mean_reading_time,...,Memory_not_dominant_hand_y_range,Memory_not_dominant_hand_z_range,Memory_dominant_hand_trigger_press_count,Memory_not_dominant_hand_trigger_press_count,Memory_dominant_hand_trigger_pressure_mean,Memory_not_dominant_hand_trigger_pressure_mean,Memory_dominant_hand_grip_count,Memory_not_dominant_hand_grip_count,Memory_dominant_hand_grip_mean,Memory_not_dominant_hand_grip_mean
0,P001,NaN,Female,Right,4,1,28.0,1,45.03,15.01,...,0.80,0.50,35,0,0.01,0.0,1275,0,0.40,0.0
1,P002,59.0,Female,Right,4,3,NaN,1,171.99,57.33,...,0.61,1.15,63,0,0.02,0.0,222,0,0.09,0.0
2,P003,82.0,Male,Right,4,4,26.0,1,338.75,112.92,...,0.75,0.61,12,0,0.00,0.0,277,0,0.09,0.0
3,P004,75.0,Male,Left,4,3,NaN,1,114.78,38.26,...,0.59,0.92,47,1,0.03,0.0,157,0,0.10,0.0
4,P005,62.0,Male,Right,4,4,27.0,1,152.90,50.97,...,0.58,0.48,28,0,0.02,0.0,125,0,0.08,0.0


In [38]:
df.describe()

,Age,Sessions_Completed_out_of_4,Help_Rating_out_of_5,MoCA_Score,Tutorial_reading_time_happened,Tutorial_total_reading_time,Tutorial_mean_reading_time,Tutorial_max_reading_time,Tutorial_median_reading_time,Tutorial_std_reading_time,...,Memory_not_dominant_hand_y_range,Memory_not_dominant_hand_z_range,Memory_dominant_hand_trigger_press_count,Memory_not_dominant_hand_trigger_press_count,Memory_dominant_hand_trigger_pressure_mean,Memory_not_dominant_hand_trigger_pressure_mean,Memory_dominant_hand_grip_count,Memory_not_dominant_hand_grip_count,Memory_dominant_hand_grip_mean,Memory_not_dominant_hand_grip_mean
count,19.000000,20.0,20.000000,13.000000,20.0,20.000000,20.000000,20.000000,20.000000,20.000000,...,20.000000,20.00000,20.000000,20.00000,20.000000,20.000000,20.000000,20.000000,20.000000,20.0
mean,71.842105,4.0,2.900000,26.769231,1.0,225.592500,75.196500,113.631500,80.527500,43.290500,...,0.546000,0.62550,169.700000,5.95000,0.070500,0.000500,361.000000,0.750000,0.163000,0.0
std,6.542707,0.0,0.788069,3.269909,0.0,105.737952,35.245713,52.048446,43.346644,25.839889,...,0.145906,0.24526,339.392507,25.90768,0.104452,0.002236,400.174567,2.572629,0.097662,0.0
min,59.000000,4.0,1.000000,17.000000,1.0,45.030000,15.010000,21.000000,17.230000,7.360000,...,0.330000,0.19000,8.000000,0.00000,0.000000,0.000000,112.000000,0.000000,0.080000,0.0
25%,69.000000,4.0,3.000000,26.000000,1.0,170.062500,56.687500,82.765000,54.715000,27.677500,...,0.417500,0.47500,13.500000,0.00000,0.010000,0.000000,151.750000,0.000000,0.100000,0.0
50%,73.000000,4.0,3.000000,27.000000,1.0,205.025000,68.340000,94.950000,70.315000,40.410000,...,0.560000,0.69000,46.500000,0.00000,0.025000,0.000000,197.500000,0.000000,0.130000,0.0
75%,75.500000,4.0,3.000000,29.000000,1.0,266.867500,88.955000,161.000000,91.415000,62.340000,...,0.632500,0.73000,109.750000,0.00000,0.060000,0.000000,274.000000,0.000000,0.185000,0.0
max,82.000000,4.0,4.000000,30.000000,1.0,534.070000,178.020000,206.770000,197.680000,91.840000,...,0.800000,1.15000,1457.000000,116.00000,0.400000,0.010000,1545.000000,11.000000,0.440000,0.0


In [39]:
df_cleaning = df.copy()

# 2. Data preprocessing

In [40]:
df_cleaning = dp.check_Nan_values(original_df= df_cleaning, missing_values_threshold=0.6)

Missing values found, number of columns with Nan values : 4
All Nan values are handled successfuly.


In [41]:
df_cleaning = dp.check_columns_type(df_without_Nan= df_cleaning)

Columns before one hot encoding : 

          Total columns :376, 
          Numerical columns : 373,  
          Categorical columns : 3, 
          Unkown columns : 0
        

 Columns after one hot encoding : 

          Total columns :375, 
          Numerical columns : 375,  
          Categorical columns : 0, 
          Unkown columns : 0
        
All Categorical columns are encoded successfuly.


In [42]:
df_cleaning = dp.check_constant_column(df_without_categorical_column=df_cleaning)

Columns before removing constant columns : 375
There are 28 constant columns, removing them...

 Columns after removing constant columns : 347
All constant columns are removed successfuly.


In [43]:
df_cleaning = dp.check_correlation(df_cleaning)

Number of columns before removing 347
No Nan correlation.
Number of small correlation 162
Number of redundant features 65
Number of total useless columns 227, Removing them...
All useless correlations are removed successfully.
Number of columns after removing 120


In [44]:
df_cleaning.head()

,Age,Help_Rating_out_of_5,MoCA_Score,Tutorial_total_reading_time,Tutorial_max_reading_time,Tutorial_intensity_reading_time,Tutorial_total_duration_hover,Tutorial_mean_duration_hover,Tutorial_max_duration_hover,Tutorial_std_duration_hover,...,Memory_Yaw_std,Memory_Pitch_std,Memory_Roll_std,Memory_Yaw_range,Memory_dominant_hand_mean_speed,Memory_not_dominant_hand_mean_speed,Memory_dominant_hand_z_range,Memory_not_dominant_hand_y_range,Gender_Male,dominant_hand_Right
0,73.0,1,28.0,45.03,21.00,0.92,20.64,1.03,2.29,0.74,...,26.23,161.72,177.36,99.05,0.16,0.03,0.82,0.80,0,1
1,59.0,3,27.0,171.99,91.56,0.98,53.05,1.02,13.42,2.25,...,47.08,165.33,172.85,180.39,0.06,0.04,1.02,0.61,0,1
2,82.0,4,26.0,338.75,155.14,0.99,145.79,2.11,20.01,4.20,...,23.27,161.90,162.61,109.38,0.06,0.01,0.98,0.75,1,1
3,75.0,3,27.0,114.78,68.19,0.97,46.90,1.47,13.24,2.97,...,25.75,171.61,101.97,234.99,0.06,0.03,0.50,0.59,1,0
4,62.0,4,27.0,152.90,88.47,0.97,54.33,1.81,11.62,2.66,...,30.07,170.64,169.17,248.93,0.12,0.02,1.05,0.58,1,1


## 2.2 Pipelines

In [ ]:
# R² = 1 - (error_model / error_baseline)
y = df_cleaning['MoCA_Score']
df_cleaning = df_cleaning.drop(columns=['MoCA_Score'])
x = df_cleaning

baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

scores = cross_val_score(baseline, x, y, cv=5)
print(scores.mean())

-2.7344832134595953


In [46]:
# Pipeline 1 — Baseline (reference)
# scaling → model
model = Ridge()

scores = cross_val_score(model, X, y, cv=5)
print(scores.mean())
scaler = StandardScaler()
y = df_cleaning['MoCA_Score']
df_cleaning = df_cleaning.drop(columns=['MoCA_Score'])
x = df_cleaning

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.4, random_state=42)
scaler.fit(X_train)


NameError: name 'X' is not defined

In [ ]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# Pipeline 2 — PCA (compression approach)
# scaling → PCA → model

In [ ]:
# Pipeline 3 — Feature selection (SelectKBest)
# scaling → SelectKBest → model


In [ ]:
# Pipeline 4 — Regularization (L1 / Lasso)
# scaling → L1 model

## 2.3 comparing pipelines
Pipeline → Cross-validation → compare scores